# Concatenate clonal summary from mc3 with 0 mutation data

In [1]:
# Execute from this cell to load the data
import pickle as pkl
import pandas as pd

# load data
data_binom = pkl.load(open("data/mc3_binom_left.pkl", "rb"))
data_Ztest = pkl.load(open("data/mc3_Ztest_left.pkl", "rb"))

In [2]:
def create_clonal_summary(data):
    """
    Create a summary table of clonality counts for each Hugo_Symbol, and this also add huge symbols with no mutations 

    Parameters:
        data (pd.DataFrame): The input DataFrame containing 'Hugo_Symbol' and 'Clonality' columns.

    Returns:
        pd.DataFrame: A summary DataFrame with counts of 'Clonal' and 'Sub-Clonal' mutations for each Hugo_Symbol.
    """
    # Group by Hugo_Symbol and count occurrences of Clonality
    clonal_summary = data.groupby("Hugo_Symbol")["Clonality"].value_counts().unstack(fill_value=0)

    # Rename columns for clarity
    clonal_summary.columns.name = None  # Remove column index name
    clonal_summary = clonal_summary.rename(columns={"Clonal": "Clonal Count", "Sub-Clonal": "Sub-Clonal Count"}).reset_index()

    # Add Hugo symbols with no mutations
    hugo_symbols = pd.read_csv("data/zero_mutation_genes.csv")
    hugo_list = hugo_symbols['hugo_symbol'].str.strip().tolist()
    df = pd.DataFrame({
        'Hugo_Symbol': hugo_list,
        'Clonal Count': [0] * len(hugo_list),
        'Sub-Clonal Count': [0] * len(hugo_list)
    })

    # Concatenate and reset index
    clonal_summary = pd.concat([clonal_summary, df], ignore_index=True)
    
    return clonal_summary

# Example usage
clonal_summary_binom = create_clonal_summary(data_binom)
clonal_summary_Ztest = create_clonal_summary(data_Ztest)